# Well-log imputation model training

Kaggle-ready workflow for tuning implemented models on validation data, evaluating selected configurations on test data, and saving the artifacts. Enable a GPU accelerator and Internet access in Kaggle.

## 1. Set up the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tranminhduc9/well-log-imputation.git"
KAGGLE = Path("/kaggle").exists()
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidates if (path / "src").is_dir()), None)

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/kaggle/working/well-log-imputation")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

PROJECT_ROOT = PROJECT_ROOT.resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

## 2. Install requirements

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

## 3. Imports and experiment settings

In [ ]:
from dataclasses import asdict
import gc
import json
import random
import shutil

import joblib
import numpy as np
import pandas as pd
import torch

from src.data.loader import DataLoader as WellLogDataLoader
from src.models.brits import BRITS, BRITSConfig
from src.models.locf import LOCF
from src.models.model import ModelConfig
from src.models.xgboost import XGBoost, XGBoostConfig
from src.preprocessing.pipeline import create_missing_mask

SEED = 912
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = (Path("/kaggle/working/well-log-results") if KAGGLE else PROJECT_ROOT / "results" / "training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

## 4. Load processed data

The data may be inside the repository or attached as a Kaggle Dataset.

In [ ]:
local_data_root = PROJECT_ROOT / "data" / "processed"
if (local_data_root / "preprocessing_parameters.json").exists():
    DATA_ROOT = local_data_root
elif KAGGLE:
    parameter_files = list(Path("/kaggle/input").rglob("preprocessing_parameters.json"))
    if not parameter_files:
        raise FileNotFoundError("Attach the processed well-log dataset to this notebook.")
    DATA_ROOT = parameter_files[0].parent
else:
    raise FileNotFoundError(f"Processed data not found at {local_data_root}")

with (DATA_ROOT / "preprocessing_parameters.json").open(encoding="utf-8") as file:
    preprocessing = json.load(file)

SCENARIOS = preprocessing["missing_scenarios"]
SEQ_LEN = preprocessing["segment_length"]
N_FEATURES = len(preprocessing["log_columns"])
train_set = WellLogDataLoader(DATA_ROOT, split="train").load()
validation_sets = {scenario: WellLogDataLoader(DATA_ROOT, split="val", scenario=scenario).load() for scenario in SCENARIOS}
test_sets = {scenario: WellLogDataLoader(DATA_ROOT, split="test", scenario=scenario).load() for scenario in SCENARIOS}

print("Data root:", DATA_ROOT)
print("Logs:", preprocessing["log_columns"])
print("Train shape:", train_set["X"].shape)

In [ ]:
def evaluate_scenarios(model, datasets):
    return {name: model.evaluate(dataset) for name, dataset in datasets.items()}


def mean_rmse(metrics):
    return float(np.mean([values["rmse"] for values in metrics.values()]))


def save_results(results):
    path = OUTPUT_DIR / "experiment_results.json"
    path.write_text(json.dumps(results, indent=2, default=str), encoding="utf-8")
    return path


COMMON_CONFIG = {"seq_len": SEQ_LEN, "n_features": N_FEATURES, "seed": SEED}
results = {}

## 5. Train, tune, and save models

Configurations are selected by mean RMSE across validation scenarios. Test data are evaluated only after selection.

### 5.1 LOCF

LOCF has no trainable parameters.

In [ ]:
locf = LOCF(ModelConfig(**COMMON_CONFIG))
locf_validation = evaluate_scenarios(locf, validation_sets)
locf_test = evaluate_scenarios(locf, test_sets)
results["locf"] = {
    "best_parameters": {},
    "validation_mean_rmse": mean_rmse(locf_validation),
    "validation": locf_validation,
    "test": locf_test,
}
save_results(results)
pd.DataFrame(locf_test).T

### 5.2 XGBoost

A small manual grid keeps the experiment understandable and affordable.

In [ ]:
XGBOOST_GRID = [
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03},
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05},
]

xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
xgb_trials = []
best_xgb = None
best_xgb_parameters = None
best_xgb_score = float("inf")

for parameters in XGBOOST_GRID:
    config = XGBoostConfig(**COMMON_CONFIG, **parameters, device=xgb_device)
    model = XGBoost(config).fit(train_set)
    validation = evaluate_scenarios(model, validation_sets)
    score = mean_rmse(validation)
    xgb_trials.append({**parameters, "validation_mean_rmse": score})
    print(parameters, "validation mean RMSE =", round(score, 6))
    if score < best_xgb_score:
        best_xgb = model
        best_xgb_parameters = parameters
        best_xgb_score = score

xgb_test = evaluate_scenarios(best_xgb, test_sets)
results["xgboost"] = {
    "best_parameters": best_xgb_parameters,
    "validation_mean_rmse": best_xgb_score,
    "trials": xgb_trials,
    "test": xgb_test,
}
joblib.dump({"config": asdict(best_xgb.config), "models": best_xgb.backend.models}, OUTPUT_DIR / "xgboost.joblib")
save_results(results)
pd.DataFrame(xgb_test).T

### 5.3 BRITS

Training segments are complete, so one reproducible mixed-scenario mask is applied. Every trial uses the same mask.

In [ ]:
BRITS_EPOCHS = 30
BRITS_GRID = [
    {"hidden_size": 64, "batch_size": 32, "learning_rate": 1e-3},
    {"hidden_size": 128, "batch_size": 32, "learning_rate": 1e-3},
    {"hidden_size": 64, "batch_size": 64, "learning_rate": 5e-4},
]

train_values = train_set["X"]
train_mask = create_missing_mask(train_values, random_state=SEED)
brits_train_values = train_values.copy()
brits_train_values[train_mask] = np.nan
brits_train_set = {"X": brits_train_values}
brits_trials = []
best_brits_parameters = None
best_brits_state = None
best_brits_score = float("inf")

for parameters in BRITS_GRID:
    config = BRITSConfig(**COMMON_CONFIG, **parameters, epochs=BRITS_EPOCHS, device=DEVICE)
    model = BRITS(config).fit(brits_train_set)
    validation = evaluate_scenarios(model, validation_sets)
    score = mean_rmse(validation)
    brits_trials.append({**parameters, "validation_mean_rmse": score})
    print(parameters, "validation mean RMSE =", round(score, 6))
    if score < best_brits_score:
        best_brits_parameters = parameters
        best_brits_score = score
        best_brits_state = {name: value.detach().cpu().clone() for name, value in model.backend.network.state_dict().items()}
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_brits_config = BRITSConfig(**COMMON_CONFIG, **best_brits_parameters, epochs=BRITS_EPOCHS, device=DEVICE)
best_brits = BRITS(best_brits_config)
best_brits.backend.network.load_state_dict(best_brits_state)
best_brits._is_fitted = True
brits_test = evaluate_scenarios(best_brits, test_sets)
results["brits"] = {
    "best_parameters": {**best_brits_parameters, "epochs": BRITS_EPOCHS},
    "validation_mean_rmse": best_brits_score,
    "trials": brits_trials,
    "test": brits_test,
}
torch.save({"config": asdict(best_brits_config), "state_dict": best_brits_state}, OUTPUT_DIR / "brits.pt")
save_results(results)
pd.DataFrame(brits_test).T

### 5.4 SAITS

`src/models/saits.py` is empty, so it is intentionally skipped. Add its training cell after the model is implemented.

In [ ]:
saits_path = PROJECT_ROOT / "src" / "models" / "saits.py"
if not saits_path.read_text(encoding="utf-8").strip():
    results["saits"] = {"status": "skipped: model is not implemented"}
    print(results["saits"]["status"])

## 6. Results and Kaggle output

In [ ]:
results_path = save_results(results)
summary = pd.DataFrame([
    {
        "model": name,
        "validation_mean_rmse": result.get("validation_mean_rmse"),
        "test_mean_rmse": mean_rmse(result["test"]) if "test" in result else None,
    }
    for name, result in results.items()
])
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Results:", results_path)
print("Downloadable archive:", archive)
summary